# Apache Spark — premiers pas

**Formation Big Data — ANSD / Data Innovation Lab**

Nous travaillons sur un fichier de recensement simulé : plusieurs millions
d'individus, une vingtaine de variables.

Le code est fourni : exécutez, observez, et modifiez les valeurs pour explorer.

Sommaire : la session · lecture et partitions · transformations et actions ·
le plan d'exécution · l'interface de suivi · SQL · le brassage.

## 1. Démarrer une session Spark

La session est le point d'entrée. Elle démarre une machine virtuelle Java :
comptez une dizaine de secondes la première fois — c'est normal.

In [ ]:
import os
import time
from pathlib import Path

from pyspark.sql import SparkSession, functions as F

DONNEES = Path(os.environ.get("DONNEES", "/travail/donnees"))
FICHIER = DONNEES / "individus.csv"

spark = (
    SparkSession.builder
    .appName("premiers_pas")
    .master("local[*]")              # tous les cœurs de cette machine
    .config("spark.sql.shuffle.partitions", "8") # nombre de partitions pour les opérations de shuffle
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("Spark", spark.version)
print("Cœurs utilisés :", spark.sparkContext.defaultParallelism)
print("Interface de suivi : http://localhost:4040")

`local[*]` demande à Spark d'utiliser tous les cœurs de la machine. Sur un
cluster, cette seule ligne changerait — et rien d'autre dans le notebook.

> Si votre poste manque de mémoire, remplacez `local[*]` par `local[2]` :
> chaque cœur mobilisé consomme de la mémoire, et il en faut aussi pour Docker
> et pour votre navigateur.

## 2. Lire un fichier

`inferSchema` demande à Spark de deviner les types en parcourant les données.
C'est pratique, mais **cela coûte une lecture complète du fichier** — avant même
d'avoir traité quoi que ce soit. Sur un fichier volumineux, on préfère déclarer
le schéma explicitement.

In [ ]:
depart = time.perf_counter()
df = spark.read.csv(str(FICHIER), header=True, inferSchema=True)
print(f"Lecture déclarée en {time.perf_counter() - depart:.1f} s")

print("Colonnes :", len(df.columns))
print("Partitions :", df.rdd.getNumPartitions())

Le temps affiché n'est pas nul, alors que nous avons dit qu'une lecture ne
déclenche rien : c'est précisément `inferSchema` qui a parcouru le fichier pour
en déduire les types.

En production, on écrit plutôt le schéma à la main — plus rapide, et sans
mauvaise surprise sur une colonne mal devinée :

```python
from pyspark.sql import types as T

schema = T.StructType([
    T.StructField("id_individu", T.IntegerType()),
    T.StructField("region",      T.StringType()),
    T.StructField("age",         T.IntegerType()),
    # …
])
df = spark.read.csv(chemin, header=True, schema=schema)
```

In [ ]:
df.printSchema()

In [ ]:
df.show(5, truncate=False)

### Les partitions

Spark a découpé le fichier en morceaux. Chaque partition sera traitée
indépendamment, par un exécuteur. Le nombre de partitions plafonne le
parallélisme : avec une seule partition, une seule tâche à la fois.

In [ ]:
# Combien de lignes dans chaque partition ?
(df.groupBy(F.spark_partition_id().alias("partition"))
   .count()
   .orderBy("partition")
   .show())

## 3. Transformations et actions

C'est le mécanisme central de Spark, et celui qui déroute au premier abord.

In [ ]:
# Ceci est une TRANSFORMATION : elle décrit un traitement, sans l'exécuter.
depart = time.perf_counter()

adultes = (
    df.filter(F.col("age") >= 15)
      .filter(F.col("age") <= 110)
      .select("region", "sexe", "age", "situation_activite")
)

print(f"Trois transformations enchaînées : {time.perf_counter() - depart:.4f} s")
print("Type de l'objet :", type(adultes).__name__)

Quelques millisecondes pour filtrer plusieurs millions de lignes ? Non :
**rien n'a été calculé**. Spark a seulement enrichi un plan.

Déclenchons maintenant le calcul avec une **action**.

In [ ]:
depart = time.perf_counter()
nombre = adultes.count()          # count() est une ACTION
print(f"count() : {time.perf_counter() - depart:.1f} s")
print(f"{nombre:,} personnes de 15 ans et plus".replace(",", " "))

| Transformations | Actions |
|---|---|
| `select` `filter` `groupBy` `join` `withColumn` `orderBy` | `count` `show` `collect` `write` `first` |
| Décrivent · ne calculent rien | Déclenchent le calcul |

**Conséquence pratique** : un enchaînement de transformations est instantané,
et toute la charge se concentre sur l'action qui suit. Si un notebook semble
bloqué, c'est presque toujours sur une action.

> ⚠️ **`collect()` ramène tout le résultat dans votre programme.** Sur un
> résultat agrégé de quelques lignes, c'est parfait. Sur une table de plusieurs
> millions de lignes, votre programme sature et Spark s'arrête. Utilisez
> `show()` pour regarder, `write` pour conserver.

## 4. Le plan d'exécution

Puisque Spark voit tout le traitement avant de l'exécuter, il le réorganise.
Regardons ce qu'il a décidé.

In [ ]:
resume = (
    df.filter(F.col("age") >= 15)
      .groupBy("region")
      .agg(
          F.count("*").alias("effectif"),
          F.avg("age").alias("age_moyen"),
      )
)

resume.explain(mode="formatted")

Trois choses à repérer dans ce plan, en le lisant **de bas en haut** :

1. `Scan csv` — la lecture. Regardez sa ligne `Output` : Spark ne lit que les
   colonnes dont il a besoin, pas les vingt et une du fichier.
2. `PushedFilters` — le filtre sur l'âge est appliqué **pendant** la lecture,
   et non après.
3. `HashAggregate` … `Exchange` … `HashAggregate` — l'agrégation se fait en
   deux temps : un calcul partiel sur chaque partition, une redistribution des
   résultats, puis une combinaison finale.

Cette étape `Exchange` porte un nom : c'est un **brassage**. Nous y revenons
plus bas.

In [ ]:
resume.show()

## 5. L'interface de suivi

Spark expose une interface web sur le port **4040**, qui montre les travaux en
cours, leur découpage en étapes et en tâches, et le temps passé.

👉 Ouvrez <http://localhost:4040> dans un onglet, puis exécutez la cellule
suivante en la regardant.

In [ ]:
# Un calcul volontairement plus long, pour avoir le temps d'observer
(df.filter(F.col("age").between(0, 110))
   .groupBy("region", "sexe", "milieu_residence")
   .agg(F.count("*").alias("effectif"))
   .orderBy(F.col("effectif").desc())
   .show(10))

Dans l'onglet **Jobs**, chaque action apparaît comme un travail. Dans
**Stages**, chaque travail est découpé en étapes, et chaque étape en tâches —
une par partition. C'est ici que le parallélisme devient visible.

## 6. Les transformations courantes

Rien de dépaysant : ce sont les opérations habituelles, avec une syntaxe propre
à Spark.

In [ ]:
# Nettoyer les libellés et écarter les âges aberrants
propre = (
    df.withColumn("region", F.initcap(F.trim(F.col("region"))))
      .filter(F.col("age").between(0, 110))
)

# Effectifs par région
(propre.groupBy("region")
       .agg(F.count("*").alias("effectif"))
       .orderBy(F.col("effectif").desc())
       .show(5))

In [ ]:
# Plusieurs indicateurs en une passe
(propre.filter(F.col("age") >= 15)
       .groupBy("milieu_residence")
       .agg(
           F.count("*").alias("effectif"),
           F.avg("age").alias("age_moyen"),
           F.sum(F.when(F.col("situation_activite").isin("Occupé", "Chômeur"), 1)
                  .otherwise(0)).alias("actifs"),
       )
       .withColumn("taux_activite",
                   F.round(100 * F.col("actifs") / F.col("effectif"), 1))
       .show())

In [ ]:
# Créer une variable dérivée : les groupes d'âges
tranches = (
    propre.withColumn(
        "groupe_age",
        F.when(F.col("age") < 15, "0-14")
         .when(F.col("age") < 35, "15-34")
         .when(F.col("age") < 60, "35-59")
         .otherwise("60+"))
)

(tranches.groupBy("groupe_age")
         .agg(F.count("*").alias("effectif"))
         .orderBy("groupe_age")
         .show())

## 7. La même chose en SQL

Une table peut être exposée comme une vue et interrogée en SQL. L'optimiseur
est le même : aucune différence de performance.

In [ ]:
propre.createOrReplaceTempView("individus")

spark.sql("""
    SELECT region,
           COUNT(*)                        AS effectif,
           ROUND(AVG(age), 1)              AS age_moyen
    FROM individus
    WHERE age >= 15
    GROUP BY region
    ORDER BY effectif DESC
    LIMIT 5
""").show()

## 8. Le brassage

Certaines opérations obligent des données à **changer de partition** : toutes
les lignes d'une même région doivent se retrouver ensemble pour être comptées.
Cette redistribution s'appelle un brassage, et c'est l'opération la plus
coûteuse de Spark — elle écrit sur disque et transite par le réseau.

In [ ]:
# Sans brassage : chaque partition est traitée indépendamment
sans = propre.filter(F.col("age") >= 60).select("region", "age")

# Avec brassage : il faut rassembler les lignes par région
avec = propre.groupBy("region").agg(F.count("*"))

for libelle, plan in [("filtre  ", sans), ("groupBy ", avec)]:
    texte = plan._jdf.queryExecution().executedPlan().toString()
    print(f"{libelle} → brassages (Exchange) : {texte.count('Exchange')}")

Le nombre de partitions produites par un brassage se règle. Par défaut
Spark en crée 200, ce qui est excessif sur un poste de travail : nous l'avons
fixé à 8 au démarrage de la session.

In [ ]:
print("Partitions après brassage :",
      spark.conf.get("spark.sql.shuffle.partitions"))

for valeur in ["4", "8", "64"]:
    spark.conf.set("spark.sql.shuffle.partitions", valeur)
    depart = time.perf_counter()
    propre.groupBy("region", "sexe").agg(F.count("*")).collect()
    print(f"  {valeur:>3} partitions → {time.perf_counter() - depart:5.1f} s")

spark.conf.set("spark.sql.shuffle.partitions", "8")

**Question.** Quel réglage est le plus rapide sur votre machine ? Que se
passerait-il avec 200 partitions, la valeur par défaut, sur un poste de
travail ?

*Votre réponse :* …

## 9. Écrire le résultat

Une écriture est une action. Spark produit un **dossier** contenant un fichier
par partition — et non un fichier unique.

In [ ]:
SORTIE = DONNEES / "resume_regions"

(propre.groupBy("region")
       .agg(F.count("*").alias("effectif"),
            F.round(F.avg("age"), 1).alias("age_moyen"))
       .write.mode("overwrite")
       .parquet(str(SORTIE)))

print("Fichiers produits :")
for fichier in sorted(SORTIE.iterdir()):
    print("  ", fichier.name)

## 10. Ce qu'il faut retenir

- Les **transformations** décrivent, les **actions** déclenchent. Tant qu'aucune
  action n'est appelée, rien n'est calculé.
- L'optimiseur ne lit que les colonnes utiles et applique les filtres pendant
  la lecture. `explain()` montre ce qu'il a décidé.
- Le **brassage** — les données qui changent de partition — est l'opération la
  plus coûteuse. `groupBy`, `join` et les tris en provoquent.
- Le nombre de partitions est un réglage à part entière.
- L'API DataFrame et le SQL donnent exactement la même exécution.
- `collect()` ramène tout dans votre programme : à réserver aux petits
  résultats.

In [ ]:
spark.stop()
print("Session arrêtée.")